## 메타퀘스트 릴레이 서버 실행

메타퀘스트 사용 시에만, 터미널을 열어 프로젝트 루트 디렉토리에서 다음 명령을 실행하세요(아래 .pem 파일 생성 방법은 README에 작성되어 있습니다).

```bash
YAM_MUJOCO_FRAME_PATH=/tmp/yam-mujoco-frame.jpg \
CAM_WIDTH=640 CAM_HEIGHT=480 CAM_FPS=15 \
.venv/bin/python -m yam_control.teleop.mujoco_relay \
  --host 0.0.0.0 \
  --port 8443 \
  --ssl-keyfile certs/key.pem \
  --ssl-certfile certs/cert.pem
```

서버가 실행되면 메타퀘스트 브라우저에서 `https://192.168.1.2:8443`(192.168.1.2 부분을 PC LAN IP에 맞게 변경해야 합니다)에 접속하세요. 아직 start teleop 버튼은 누르지 마세요.

## 1. 실행 설정

In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import display

# Quest stream용 MuJoCo offscreen renderer에 EGL backend를 사용
os.environ.setdefault('MUJOCO_GL', 'egl')

PROJECT_ROOT: Path = Path.cwd().resolve()
YAM_ABC_ROOT: Path = PROJECT_ROOT / 'third_party' / 'yam-abc-reproduce'
I2RT_ROOT: Path = YAM_ABC_ROOT / 'third_party' / 'i2rt'
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(YAM_ABC_ROOT))
sys.path.insert(0, str(I2RT_ROOT))

from yam_control import RunConfig, RuntimeDependencies, build_run_config, create_session
from yam_control.config import CameraConfig, ExecutionTarget, QuestConfig, QuestControllerHand, QuestDisplayMode, RobotConfig, RunMode, TeleopSourceType, VLAType
from yam_control.session import RunSession
from yam_control.types import EpisodeState

MODE: RunMode = 'teleop'

TELEOP_SOURCE: TeleopSourceType = 'quest3'

VLA_TYPE: VLAType = 'pi0.5'

EXECUTION_TARGET: ExecutionTarget = 'real'

TASK_PROMPT: str = 'pick up the object'

SAVE_TELEOP_DATA: bool = True

USE_SAFETY_GATE: bool = False

USE_RTC: bool = False

# 1초 = 30스텝
EPISODE_STEPS: int = 30 * 120

SIMULATION_EPISODES: int = 1

QUEST_TRANSLATION_SCALE: float = 0.8

QUEST_ROTATION_SCALE: float = 1.0

QUEST_POSITION_REACH_LIMIT_M: float = 0.10

QUEST_ROTATION_REACH_LIMIT_RAD: float = 0.35

QUEST_MAX_JOINT_DELTA_RAD: float = 0.1

# IK 위치 cost 1.0 대비 방향 cost, 낮을수록 손목 방향보다 손 위치를 우선
QUEST_IK_ORIENTATION_COST: float = 0.5

QUEST_STREAM_WIDTH: int = 640

QUEST_STREAM_HEIGHT: int = 480

QUEST_STREAM_FPS: float = 15.0

QUEST_STREAM_JPEG_QUALITY: int = 70

CHECKPOINT_URI: str = ''

CHECKPOINT_REVISION: str | None = None

POLICY_CONFIG_NAME: str = ''

CONTROL_HZ: float = 30.0

FOLLOWER_CAN_CHANNEL: str = 'can0'

LEADER_CAN_CHANNEL: str = 'can1'

GRIPPER_TYPE: str = 'linear_4310'

EPISODE_INITIAL_POSE: tuple[float, ...] = (0.0, 0.8, 0.8, 0.0, 0.0, 0.0, 1.0)

HUMAN_RESET_POSE: tuple[float, ...] | None = None

QUEST_RELAY_HOST: str = '127.0.0.1'

QUEST_RELAY_PORT: int = 8443

QUEST_CONTROLLER_HAND: QuestControllerHand = 'right'

QUEST_DISPLAY_MODE: QuestDisplayMode = 'robot_camera'

QUEST_STREAM_FRAME_PATH: str = '/tmp/yam-mujoco-frame.jpg'

DATA_ROOT: str = 'data/episodes'

robot_config: RobotConfig = RobotConfig(
    follower_can_channel=FOLLOWER_CAN_CHANNEL,
    leader_can_channel=LEADER_CAN_CHANNEL,
    gripper_type=GRIPPER_TYPE,
    episode_initial_pose=EPISODE_INITIAL_POSE,
    human_reset_pose=HUMAN_RESET_POSE,
)
quest_config: QuestConfig = QuestConfig(
    relay_host=QUEST_RELAY_HOST,
    relay_port=QUEST_RELAY_PORT,
    display_mode=QUEST_DISPLAY_MODE,
    controller_hand=QUEST_CONTROLLER_HAND,
    translation_scale=QUEST_TRANSLATION_SCALE,
    rotation_scale=QUEST_ROTATION_SCALE,
    position_reach_limit_m=QUEST_POSITION_REACH_LIMIT_M,
    rotation_reach_limit_rad=QUEST_ROTATION_REACH_LIMIT_RAD,
    max_joint_delta_rad=QUEST_MAX_JOINT_DELTA_RAD,
    ik_orientation_cost=QUEST_IK_ORIENTATION_COST,
    stream_frame_path=QUEST_STREAM_FRAME_PATH,
    stream_width=QUEST_STREAM_WIDTH,
    stream_height=QUEST_STREAM_HEIGHT,
    stream_fps=QUEST_STREAM_FPS,
    stream_jpeg_quality=QUEST_STREAM_JPEG_QUALITY,
)
config: RunConfig = build_run_config(
    mode=MODE,
    teleop_source=TELEOP_SOURCE,
    save_teleop_data=SAVE_TELEOP_DATA,
    vla_type=VLA_TYPE,
    checkpoint_uri=CHECKPOINT_URI,
    checkpoint_revision=CHECKPOINT_REVISION,
    policy_config_name=POLICY_CONFIG_NAME,
    use_rtc=USE_RTC,
    execution_target=EXECUTION_TARGET,
    use_safety_gate=USE_SAFETY_GATE,
    control_hz=CONTROL_HZ,
    task_prompt=TASK_PROMPT,
    data_root=DATA_ROOT,
    robot=robot_config,
    quest=quest_config,
    # Camera는 '2. 카메라 연결' cell에서 설정
    camera=CameraConfig(),
)

## 2. 카메라 연결

카메라 설정을 `config`에 적용하고, 카메라를 열어 224x224로 변환한 프레임 하나를 표시한 뒤 닫습니다. 실제 로봇 실행 시에는 `session.connect()`가 카메라를 다시 엽니다.

In [ ]:
from PIL import Image
from yam_abc_reproduce.camera.interface import CameraFrame

from yam_control.camera import CameraRig
from yam_control.config import CameraDeviceConfig, CameraFitMode, with_camera_config

TOP_CAMERA_DEVICE_PATH: str = '/dev/v4l/by-id/usb-Intel_R__RealSense_TM__Camera_Intel_R__RealSense_TM__515_00000000F0244932-video-index0'

TOP_CAMERA_CAPTURE_WIDTH: int = 960

TOP_CAMERA_CAPTURE_HEIGHT: int = 540

TOP_CAMERA_CAPTURE_FPS: int = 30

TOP_CAMERA_PIXEL_FORMAT: str = 'YUYV'

# 'center_crop'은 좌우를 잘라 가운데만 남기고 'zero_pad'는 위아래에 검은 영역을 채움
TOP_CAMERA_FIT_MODE: CameraFitMode = 'center_crop'

CAMERA_OUTPUT_SIZE: int = 224

CAMERA_SETTLE_TIME_S: float = 1.0

CAMERA_MAX_FRAME_AGE_S: float = 0.5

# 손목 camera를 추가하면 role='wrist'인 CameraDeviceConfig를 이 tuple에 추가
camera_devices: tuple[CameraDeviceConfig, ...] = (
    CameraDeviceConfig(
        role='top',
        device_path=TOP_CAMERA_DEVICE_PATH,
        capture_width=TOP_CAMERA_CAPTURE_WIDTH,
        capture_height=TOP_CAMERA_CAPTURE_HEIGHT,
        capture_fps=TOP_CAMERA_CAPTURE_FPS,
        pixel_format=TOP_CAMERA_PIXEL_FORMAT,
        fit_mode=TOP_CAMERA_FIT_MODE,
        output_size=CAMERA_OUTPUT_SIZE,
    ),
)
camera_config: CameraConfig = CameraConfig(
    devices=camera_devices,
    settle_time_s=CAMERA_SETTLE_TIME_S,
    max_frame_age_s=CAMERA_MAX_FRAME_AGE_S,
)
config = with_camera_config(config, camera_config)

# 연결 확인용으로 camera를 열어 frame 하나를 읽고 해제
camera_rig: CameraRig = CameraRig(camera_config)
camera_rig.connect()
try:
    camera_frames: dict[str, CameraFrame] = camera_rig.read_frames()
finally:
    camera_rig.close()
camera_role: str
camera_frame: CameraFrame
for camera_role, camera_frame in camera_frames.items():
    display({'role': camera_role, 'shape': camera_frame.images['rgb'].shape})
    display(Image.fromarray(camera_frame.images['rgb']))

## 3. Session 생성

In [ ]:
dependencies: RuntimeDependencies = RuntimeDependencies()
session: RunSession = create_session(
    config=config,
    dependencies=dependencies,
)

## 4. 메타퀘스트, MuJoCo 연결

메타퀘스트 웹브라우저에서 start teleop 버튼을 누른 뒤에 아래 셀을 실행시키세요. 메타퀘스트와 MuJoCo backend를 연결하고 첫 유효 `xr_frame`을 최대 10초 기다립니다.

In [ ]:
# 선택된 로봇과 action source를 연결
session.connect()

## 5. Episode 초기화

로봇을 `episode_initial_pose`로 보내고 에피소드 시작 상태를 준비합니다. 그리퍼를 완전히 열기 때문에 그리퍼가 책상을 누르는 상태에서는 실행시키면 안됩니다.

In [ ]:
# 실제 로봇은 human reset pose로 이동하고 MuJoCo는 environment를 자동 reset
session.prepare_episode()

## 6. Teleoperation 실행

에피소드를 시작합니다. (메타퀘스트 사용 시)리모컨의 Grip(중지로 누르는 버튼)을 누르는 동안만 로봇이 움직입니다.

`SAVE_TELEOP_DATA = True`이면 Grip을 누르고 있는 구간만 `DATA_ROOT`에 저장되고, Grip을 놓은 구간은 저장하지 않습니다. MuJoCo에서는 `SAVE_TELEOP_DATA` 값과 관계없이 저장하지 않습니다.

에피소드 도중 리모컨의 A 버튼(왼손 리모컨 사용 시 X)을 누르면 성공 에피소드로 보고 즉시 종료한 뒤 저장합니다(`SUCCEEDED`). B 버튼(왼손 리모컨 사용 시 Y)을 누르면 실패 에피소드로 보고 즉시 종료하며 저장하지 않습니다(`DISCARDED`). 버튼을 누르지 않고 `EPISODE_STEPS`에 도달하면 저장하고 종료합니다(`FINISHED`).

### 저장되는 값

에피소드마다 `DATA_ROOT/<task>/<시각_id>/` 폴더에 저장됩니다. `N`은 저장된 step 수입니다(Grip을 놓은 step은 제외).

| 파일 | shape | 내용 |
|---|---|---|
| `left-joint_pos.npy` | `(N, 6)` | 각 step 시작 시 측정한 팔 관절 각도(rad) |
| `left-gripper_pos.npy` | `(N, 1)` | 측정한 그리퍼 개폐 정도(0 닫힘 ~ 1 열림) |
| `left-ee_pose.npy` | `(N, 7)` | 측정 관절 각도로 FK한 `grasp_site` pose `(x, y, z, qx, qy, qz, qw)` |
| `action-left-joint.npy` | `(N, 6)` | 해당 step에 로봇에 명령한 관절 목표 각도(rad, 절대값) |
| `action-left-gripper.npy` | `(N, 1)` | 명령한 그리퍼 값(`1 - 트리거`) |
| `action-left-ee_pose.npy` | `(N, 7)` | 명령 관절 각도로 FK한 `grasp_site` pose `(x, y, z, qx, qy, qz, qw)` |
| `<role>-images-rgb.mp4` | `N` frames | 카메라 RGB 영상(224x224, H.264), 카메라 셀의 crop/pad 설정 적용 |
| `<role>-timestamp.npy` | `(N,)` | 각 frame의 카메라 수신 시각(ms, Unix time) |
| `metadata.json` | - | task 이름, arm 이름, 관절 수, `control_hz`, 카메라 정보, frame 수, EE pose 형식(`extra.ee_pose`) |
| `write_complete.flag` | - | 저장이 끝까지 완료되었음을 표시 |

- EE pose 기준 좌표계는 **로봇 베이스 좌표계**(MuJoCo YAM model의 world 좌표계와 동일)입니다.
  - 원점: 로봇 베이스 바닥면 중심. 베이스 회전축(joint1)이 이 점을 수직으로 지나며, joint1은 원점에서 6.8 cm 위에 있습니다.
  - +x: 로봇 정면(모든 관절이 0일 때 팔이 뻗는 방향), +y: 로봇 기준 왼쪽, +z: 위쪽
  - 위치 단위는 m이고, 로봇을 테이블 위에서 옮겨도 이 좌표계는 로봇을 따라 움직입니다(테이블이나 카메라 기준 좌표가 아님).
  - 예: `EPISODE_INITIAL_POSE` = (0.0, 1.2, 0.9, 0.0, 0.0, 0.0, 1.0)의 EE 위치는 (0.410, 0.000, 0.272) m로, 베이스 앞쪽 41 cm, 높이 27 cm입니다.
- EE pose 방향은 `grasp_site` 좌표축 기준입니다. z축은 그리퍼가 뻗은 방향(접근 방향), y축은 두 손가락이 열리고 닫히는 방향입니다.
- `grasp_site`는 손목 flange에서 그리퍼 축을 따라 134.7 mm 떨어진 점으로, 손가락 끝에서 약 1 cm 안쪽의 두 손가락 중심점이며 텔레오퍼레이션 IK가 따라가는 점과 같습니다.
- Quaternion 부호는 에피소드 안에서 연속되도록 맞춰 저장합니다(첫 frame은 `qw >= 0`).
- 카메라 frame은 관절 측정 시점의 최신 frame이며 최대 약 33 ms 먼저 찍혔을 수 있습니다.

In [ ]:
# 실제 로봇은 episode 하나를 실행하고 MuJoCo는 여러 에피소드를 연속 실행
episode_result: EpisodeState | tuple[EpisodeState, ...]
if config.common.execution_target == 'real':
    episode_result = session.run_prepared_episode(max_steps=EPISODE_STEPS)
else:
    episode_result = session.run_simulation_episodes(
        episode_count=SIMULATION_EPISODES,
        max_steps=EPISODE_STEPS,
    )
display(episode_result)
display(session.action_producer_diagnostics())
display({'control_loop': session.control_loop_diagnostics()})

## 7. 그리퍼 열기

아래 셀을 실행하면 로봇이 현재 자세를 유지하면서 그리퍼만 엽니다. 로봇이 잡고 있는 물건을 빼야 할 때 사용하는 셀입니다.

In [ ]:
# 현재 측정 자세에서 gripper만 완전히 연 pose로 이동
GRIPPER_OPEN: float = 1.0

current_state: tuple[float, ...] = session._robot.get_observation().state
open_gripper_pose: tuple[float, ...] = (*current_state[:6], GRIPPER_OPEN)
display({'current': current_state, 'target': open_gripper_pose})
session._robot.move_to_pose(open_gripper_pose)

## 8. Rest pose

로봇을 rest pose로 이동합니다. 데이터 수집을 마친 뒤, 혹은 주피터 노트북 커널을 restart하기 전에 이 셀을 실행해주시면 로봇이 낙하로 인한 충격을 받지 않습니다.

In [ ]:
REST_POSE: tuple[float, ...] = (0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 1.0)
session._robot.move_to_pose(REST_POSE)

## 9. Session 종료

메타퀘스트 controller reader, MuJoCo viewer와 session resource를 안전하게 해제합니다.

In [ ]:
# Controller, VLA와 robot resource를 해제
session.close()